# 22 · Fresh external-cohort addendum without retuning

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run after the main analysis lock and fresh collection. It does not rebuild or overwrite the main split or vocabularies.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Ingest a post-lock cohort under the frozen protocol

In [ ]:
from oncoplate.external import prepare_external_cohort,external_frozen_predictions
COHORT_ID='prospective_v1'
IMPORT_ROOT=Path(cfg['root'])/'data/imports'/COHORT_ID
GROUP_REVIEW_REFERENCE=''  # Actual review of duplicates and acquisition/product-family links.
external_root=prepare_external_cohort(cfg,COHORT_ID,IMPORT_ROOT,GROUP_REVIEW_REFERENCE)
print(external_root)

## 2. Apply the already frozen predictors and policies

In [ ]:
pred=external_frozen_predictions(cfg,COHORT_ID)
print('Predictions:',len(pred),'Independent rating jobs:',external_root)

## 3. Join real adjudications and compute paired external results

In [ ]:
from oncoplate.governance import verify_lock
from oncoplate.benchmark import adjudicated_ratings
from oncoplate.evaluation import attach_outcomes,evaluate_results
from oncoplate.statistics import paired_bootstrap
import pandas as pd
protocol=verify_lock(p['private']/'analysis_lock.json')['protocol'];frames=[]
for run_id in protocol['run_ids']:
    pp=read_table(external_root/f'{run_id}_frozen_policy_predictions.csv');pp['seed']=pp.seed.astype(int)
    rr=adjudicated_ratings(read_table(external_root/f'{run_id}_ratings_adjudicated.csv'))
    frames.append(attach_outcomes(pp,rr))
results=pd.concat(frames,ignore_index=True)
write_table(external_root/'evaluation_results.csv',results)
table=evaluate_results(results,protocol['domain_mixture']);write_table(p['reports']/'external_selective_results.csv',table)
summary,rep=paired_bootstrap(results,mixture=protocol['domain_mixture'],seeds=protocol['model_seeds'],B=1000)
write_json(p['reports']/'external_paired_comparison.json',summary);display(table)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
